# Dense passage retrieval with Transformer encoders

This notebook shows how to use the code in `src/` and produces the tables and figures in the report. Run it from the repository root with the project's virtual environment.

Sections 1 to 5 call the code directly: data loading, TF-IDF and BM25, the evaluation metrics, the Transformer encoder, the DPR loss and trainer, and the vector index. Sections 6 to 8 analyse the experiments. They read the logs and per-query rankings that `bash run_experiments.sh` writes to `results/`, and sections 5 and 8 load trained encoder weights.

Figures go to `results/figures/` and LaTeX tables to `results/tables/`.

In [ ]:
import ast
import math
import random
import re
import time
import json
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from hydra import compose, initialize_config_dir
from matplotlib.colors import PowerNorm
from matplotlib.ticker import MaxNLocator
from torch.utils.data import DataLoader

from src.data_utils.corpus import Corpus
from src.data_utils.query import Query, make_queries
from src.evaluation.attention_maps import get_attention_maps
from src.evaluation.metrics import Evaluate
from src.retrievers.bm25 import BM25
from src.retrievers.dpr import DPR
from src.retrievers.models.layers import attention
from src.retrievers.tf_idf import TFIDF
from src.retrievers.utils import retrieveQueryAndGetRelevancies
from src.retrievers.vector_index import VectorIndex
from src.training.loss import DPRLoss
from src.training.trainer import PairDataset, Trainer

RESULTS = Path('results')
SCRATCH_CKPT = RESULTS / 'tweaks/warmup_170/rep1'
BERT_CKPT = RESULTS / 'bert/cls/rep1'
FIGURES = RESULTS / 'figures'
TABLES = RESULTS / 'tables'
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)




initialize_config_dir(config_dir=str(Path('config').resolve()), version_base=None)
pd.set_option('display.max_colwidth', 120)
plt.rcParams.update({'font.size': 8, 'axes.titlesize': 8, 'axes.labelsize': 8, 'legend.fontsize': 7,
                     'xtick.labelsize': 7, 'ytick.labelsize': 7, 'axes.spines.top': False,
                     'axes.spines.right': False, 'savefig.bbox': 'tight', 'figure.dpi': 120})
COLORS = {'TF-IDF': '#999999', 'BM25': '#E69F00', 'DPR scratch': '#56B4E9', 'DPR BERT': '#0072B2',
          'BM25 + DPR BERT': '#009E73'}


def config(*overrides, ckpt=None):
    overrides = list(overrides)
    if ckpt is not None:
        overrides += [f'+model.query_encoder_weights_path={ckpt}/query_encoder.pt',
                      f'+model.document_encoder_weights_path={ckpt}/document_encoder.pt']
    return compose(config_name='fiqa_experiment',overrides=overrides)


def words(text):
    return TFIDF.PreprocessText(None, text)


def shorten(text, n=160):
    text = ' '.join(text.split())
    return text if len(text) <= n else text[:n] + '...'


def save_table(df, name):
    rows= [' & '.join([df.index.name or ''] + list(df.columns)) + r' \\', r'\midrule']
    rows += [' & '.join([str(i)] + [str(v) for v in row]) + r' \\' for i, row in zip(df.index,df.values)]
    body = [r'\begin{tabular}{l' + 'c' * len(df.columns) + '}', r'\toprule'] + rows + [r'\bottomrule',r'\end{tabular}']
    (TABLES / f'{name}.tex').write_text('\n'.join(body).replace('±', r'$\pm$').replace('%', r'\%') + '\n')

## 1. Data

`Corpus` downloads FiQA and SQuAD once, assigns query and passage ids, and caches the result in `retrieval_data/`. Each split holds queries with their relevant passages. The retrieval collection (`all_documents`) is every unique passage across the splits, so a SQuAD paragraph asked about by five questions is a single passage. SQuAD has no test split.

In [ ]:
corpora = {name: Corpus(dataset_name=name,base_dir='./retrieval_data') for name in ('fiqa', 'squad')}
fiqa, squad = corpora['fiqa'], corpora['squad']


def dataset_statistics(corpus):
    splits = [s for s in (corpus.train, corpus.val,corpus.test) if s is not None]
    lengths = [len(words(texts[0])) for texts in corpus.all_documents['text']]
    return {'train queries': len(corpus.train),
            'validation queries': len(corpus.val),
            'test queries': len(corpus.test) if corpus.test is not None else 0,
            'passages': len(lengths),
            'relevant passages per query': np.mean([len(ids) for s in splits for ids in s['document_id']]),
            'words per query': np.mean([len(words(q)) for s in splits for q in s['question']]),
            'words per passage (mean)': np.mean(lengths),
            'words per passage (median)': np.median(lengths)}


statistics = pd.DataFrame({'FiQA': dataset_statistics(fiqa), 'SQuAD': dataset_statistics(squad)})
statistics.round(1)

In [ ]:
example = fiqa.test[0]
print('Query:', example['question'])
print(f"{len(example['text'])} relevant passages, the first one:")
print(shorten(example['text'][0], 400))

## 2. Lexical retrieval with TF-IDF and BM25

Both retrievers build a sparse passage-by-term matrix when they are created. Scoring a batch of queries is then one sparse matrix product with the queries' term-count vectors. Text is lower-cased and stripped of non-alphanumeric characters. There is no stemming or stop-word removal.

In [ ]:
start = time.time()
tfidf = TFIDF(documents=fiqa.all_documents)
bm25 = BM25(documents=fiqa.all_documents)
print(f'Built TF-IDF and BM25 in {time.time() - start:.0f} s; vocabulary of {len(bm25.corpus_vocabulary):,} terms')

In [ ]:
terms= ['the', 'money', 'tax', 'mortgage', 'dividend', 'quickbooks']
pd.DataFrame({'passages containing term': [sum(term in set(words(d.GetText())) for d in bm25.index.GetDocuments()) for term in terms],'TF-IDF IDF': [tfidf.idf[t] for t in terms],'BM25 IDF': [bm25.idf[t] for t in terms]}, index=terms).round(2)

In [ ]:
QUERY = 'how do I deduct business travel expenses'


def show(retriever, query, k=3):
    for rank, doc in enumerate(retriever.Lookup([query], k=k)[0],start=1):
        print(f'{rank}. {shorten(doc.GetText())}')


for name, retriever in [('TF-IDF', tfidf), ('BM25', bm25)]:
    print(name)
    show(retriever, QUERY)

### Evaluation metrics

`retrieveQueryAndGetRelevancies` returns, for every query, a list of booleans: whether each retrieved passage is relevant. `Evaluate` turns these lists into P@k, R@k and MRR, where k is the length of the lists. It reports two recalls. `recall_standard` is hits divided by the number of relevant passages. `recall` divides by min(k, number relevant) instead, so it can reach 1 at k=1. The report uses `recall_standard`, because the capped version can decrease as k grows.

In [ ]:
toy_queries = [Query('first query', id='q0',relevant_document_ids=['d1', 'd2']),
               Query('second query', id='q1', relevant_document_ids=['d3'])]
toy_relevancies = [[False, True, False],
                   [False,False, False]]
Evaluate(toy_relevancies, toy_queries)

In [ ]:
fiqa_val = make_queries(fiqa, fiqa.val)
fiqa_test = make_queries(fiqa, fiqa.test)
for name, retriever in [('TF-IDF', tfidf), ('BM25', bm25)]:
    relevancies = retrieveQueryAndGetRelevancies(retriever, fiqa_val,k=10)
    print(name, {metric: round(value, 3) for metric,value in Evaluate(relevancies, fiqa_val).items()})

### BM25 parameters and passage length

$k_1$ controls how quickly repeated query terms stop adding to the score, and $b$ controls length normalisation. We rebuild the BM25 matrix for a few settings and evaluate on FiQA validation. We also record the mean length of the top-ranked passage, which shows how each setting treats long passages.

In [ ]:
passage_length = {doc.GetId(): len(words(doc.GetText())) for doc in bm25.index.GetDocuments()}


def evaluate_bm25(k1, b):
    bm25.k1, bm25.b = k1, b
    bm25.bm25_matrix = bm25.GetDocumentsTFIDFVectors()
    relevancies, retrieved = retrieveQueryAndGetRelevancies(bm25, fiqa_val, k=10, return_documents=True)
    scores = Evaluate(relevancies, fiqa_val)
    return {'k1': k1, 'b': b, 'R@10': scores['recall_standard'], 'MRR': scores['mrr'],
            'top-1 passage words': np.mean([passage_length[docs[0].GetId()] for docs in retrieved])}


settings = [(1.5, b) for b in (0.0, 0.25, 0.5,0.75, 1.0)] + [(k1, 0.75) for k1 in (0.5, 1.0, 2.0, 3.0)]
sweep = pd.DataFrame([evaluate_bm25(k1, b) for k1, b in settings])
bm25.k1, bm25.b = 1.5, 0.75
bm25.bm25_matrix = bm25.GetDocumentsTFIDFVectors()
sweep.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(3.4, 1.6),sharey=True)
for ax, (param, label, rows) in zip(axes, [('b', '$b$ ($k_1=1.5$)', sweep.k1 == 1.5),
                                           ('k1', '$k_1$ ($b=0.75$)', sweep.b == 0.75)]):
    part = sweep[rows].sort_values(param)
    ax.plot(part[param], part['R@10'], 'o-', color=COLORS['BM25'],ms=3, label='R@10')
    ax.plot(part[param], part['MRR'], 's--', color='#555555', ms=3, label='MRR')
    ax.set_xlabel(label)
axes[0].set_ylabel('FiQA validation')
axes[1].legend(frameon=False, loc='lower right')
fig.savefig(FIGURES / 'bm25_sweep.pdf')

## 3. The Transformer encoder

`TransformerEncoder` (`src/retrievers/models/encoder.py`) embeds WordPiece tokens, adds sinusoidal positional encodings, applies a stack of encoder layers (multi-head self-attention and a feed-forward block, each with a residual connection and layer normalisation), pools the token vectors into one vector and projects it to the output dimension. `PretrainedEncoder` has the same interface around a Hugging Face model, so `DPR` can use either one. The model is chosen with a Hydra override such as `model=bert`.

In [ ]:
cfg_scratch = config()
dpr_scratch = DPR(cfg_scratch, documents=fiqa.all_documents)
encoder = dpr_scratch.query_encoder

ids = dpr_scratch.tokenize([QUERY, 'what is a roth ira'], cfg_scratch.model.query_max_length)
print('token ids:',tuple(ids.shape))
print(dpr_scratch.tokenizer.convert_ids_to_tokens(ids[0]))
print(dpr_scratch.tokenizer.convert_ids_to_tokens(ids[1]))
with torch.no_grad():
    print('encoder output:',tuple(encoder(ids.to(dpr_scratch.device)).shape))

In [ ]:
cfg_bert = config('model=bert')
dpr_bert_pretrained = DPR(cfg_bert,documents=fiqa.all_documents)
n_parameters = lambda module: sum(p.numel() for p in module.parameters())
pd.Series({'scratch encoder': n_parameters(encoder),
           'of which token embeddings': encoder.token_embedding.weight.numel(),
           'BERT-base encoder': n_parameters(dpr_bert_pretrained.query_encoder)}).map('{:,}'.format)

Scaled dot-product attention with a padding mask. The last two keys are padding, so every query gives them zero weight and each row still sums to one.

In [ ]:
torch.manual_seed(0)
x = torch.randn(1, 5, 8)
padding_mask = torch.tensor([[[1, 1, 1, 0, 0]]])
_, weights = attention(x, x,x,mask=padding_mask)
print(weights[0].numpy().round(2))
print('row sums:', weights[0].sum(-1).numpy().round(3))

The three pooling options on a toy sequence of three tokens plus one padding position.

In [ ]:
token_vectors = torch.tensor([[[1., 2.], [3., 4.], [5., 0.],[9.,9.]]])
token_mask = torch.tensor([[[True, True, True, False]]])
for pooling in ('cls', 'mean', 'max'):
    encoder.pooling_type = pooling
    print(f'{pooling:>4}:', encoder.pool(token_vectors, token_mask).tolist())
encoder.pooling_type = 'mean'

## 4. The DPR training setup

`DPRLoss` scores every query in a batch against every passage in the batch and applies cross-entropy with the matching passage as the target. The other passages in the batch act as negatives. If the encoders carry no information, all scores are equal and the loss is $\ln B$ for batch size $B$. This is the chance level in the training curves.

In [ ]:
criterion = DPRLoss()
torch.manual_seed(0)
B = 8
queries = torch.randn(B, 32)
print(f'equal scores:   {criterion(torch.zeros(B, 32),torch.zeros(B, 32)).mean():.3f}  (ln B = {math.log(B):.3f})')
print(f'random vectors: {criterion(queries, torch.randn(B, 32)).mean():.3f}')
print(f'matched pairs:  {criterion(queries, queries + 0.1 * torch.randn(B, 32)).mean():.3f}')

`Trainer` wraps the two encoders, AdamW, an optional linear warm-up and gradient clipping. `PairDataset` draws one relevant passage per query each epoch. A few epochs on 1,024 training pairs show the loss falling below $\ln 32 = 3.47$. The real runs (`main.py` with `do_train=true`) add early stopping on the validation loss and keep the best checkpoint.

In [ ]:
random.seed(0)
torch.manual_seed(0)
cfg_demo = config('train.batch_size=32')
dpr_demo = DPR(cfg_demo, documents=fiqa.all_documents)
trainer = Trainer(dpr_demo, cfg_demo)
loader = DataLoader(PairDataset(fiqa.train.select(range(1024))),batch_size=32,shuffle=True,drop_last=True)
for epoch in range(5):
    print(f'epoch {epoch}: train loss {trainer.train_epoch(loader):.3f}')